# M5P1 - IGV Usage

In this practice, we will install IGV within an environment on the HellBender Desktop. Navigation of the IGV browser will also be demonstrated. Below is a video lecture, as well as some helpful tools and explanations within the practice.

Objectives: 
- Understand transcript splicing, isoforms, and junctions
- Install IGV within an environment on the Hellbender Desktop
- Launch IGV, load the TAIR10 genome annotation, and load files

```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

<iframe width="560" height="315" src="https://www.youtube.com/embed/X4EIaXVi2UQ?si=xre7xIm600ZHMuIC" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

## Review of Concepts: 

### Transcript splicing: 

Transcript splicing is the process by which introns are removed from a newly made pre-mRNA transcript and exons are joined together to form a mature mRNA.
*[Third-party figure omitted from the public version; original source not confirmed.]*
- Genes in eukaryotes contain exons (expressed sequences) and introns (intervening sequences).
- The spliceosome, a large RNA–protein machine, recognizes conserved sequences:
    - 5′ splice site (donor)
    - 3′ splice site (acceptor)
    - Branch point
- The spliceosome cuts out the intron and ligates exons together in the process of creating an mRNA transcript, which is what our reads are sequenced from (well, the cDNA conversion of these mRNA transcripts).

In RNA-Seq data, Splicing patterns determine which exons appear in the final transcript. This means Splicing errors can reflect misalignment, novel isoforms, or low-quality RNA.

### Transcript isoforms: 

Transcript isoforms are different mRNA products created from the same gene via alternative usage of exons, splice sites, or transcription start/termination sites. This means the same gene can encode multiple functions. When the mRNA is spliced, different isoforms create different proteins that have varying functions. 

*[Third-party figure omitted from the public version; original source not confirmed.]*

alternate splicing events include: 
- Exon skipping: An exon is included in some transcripts and excluded in others.
- Alternative 5′ or 3′ splice sites: Different donor sites produce variable exon start points.
- Mutually exclusive exons
- Intron retention: very common in plants, including Arabidopsis.
- Alternative transcription start sites (TSS)
- Alternative polyadenylation

Many plant genes produce multiple functional proteins from the same gene. This isoform switching often occurs under stress, during development, or due to mutations. In IGV, isoforms appear as multiple transcript models under the same gene name within the annotation track.

### Splice junctions: 

*[Third-party figure omitted from the public version; original source not confirmed.]*

Splice junctions are the boundaries where two exons are joined together after an intron is removed. In our sequencing data processed by STAR, we can see these splice junctions in the `*_SJ.Out.tab` file. These "Junction Reads" show reads where the alignments (concatenations of reads) span across multiple exons. We can use this to investigate the splicing structure of our isoforms, identify novel isoforms caused by a stress response, and detect disruption in splicing occurring in mutant organisms. 


## Installing IGV
IGV (Integrative Genomics Viewer) is a desktop genome browser for visualizing alignments, gene models, coverage, variants, and splicing events. For RNA-Seq it allows you to inspect mapping quality, exon usage, isoforms, junctions, and expression differences across samples.

We will make an environment called `module_5_IGV` from the `Module_5_IGV.yml` file within `Module_5/Resources/`. Below is a video tutorial on how to launch IGV within the Hellbender desktop:

# Opening IGV: 
To launch IGV, you can open a terminal, load the mamba module, activate the `module_5_IGV` environment, and then type the `igv` command: 

    #!bin/bash
    
    ml mamba
    
    source activate module_5_IGV
    
    igv


# IGV file formats: 

To view RNA-Seq data, you will need:

Alignment files

- `*.bam` — coordinate-sorted read alignments

- `*.bam.bai` — BAM index (samtools index)

Coverage or expression tracks

- `*.bigwig` — per-base coverage files (recommended for RNA-Seq when we are not looking at splicing events and just want to see expression)

- `*.bed`, `*.bedgraph` — region-based tracks

Splice junctions

- `SJ.out.tab` from STAR, or

- `junctions.bed` from regtools

Annotation

`*.gtf` or `*.gff3` — gene models displayed alongside alignments. We will use TAIR10 today, these also need to be indexed if you are importing from `.gtf`.

## Preparing inputs for IGV: 
When you have a STAR alignment, you need to process the files so IGV can display them correctly. We have generated our outputs within M5L3, and they are located in the `Module_5/m5_project_STAR/results/alignments/` folder. 

To prepare these inputs, we will need to create an index file (`.bai`) from our `Aligned.sortedByCoord.out.bam` file for each sample: 

    samtools index STAR_sorted.bam

Then we can create a coverage track using `bamCoverage`

    bamCoverage -b STAR_sorted.bam \
            -o STAR_sorted.bw \
            --normalizeUsing CPM \
            --binSize 10 \
            --extendReads  # optional, extend reads to fragment length

Finally, we will convert our splice junctions to an IGV usable format (`.bed`) with regtools from our `bam` files: 

    regtools junctions extract \
    -a 8 \       # minimum anchor length on each side
    -m 50 \      # minimum intron length
    -M 500000 \  # maximum intron length
    -s 0 \       # 0 = unstranded; change to 1 or 2 for stranded libraries
    $BAM \
    -o STAR_junctions.bed

Alternatively, we could convert our `SJ.out.tab` file from star into bed format pretty easily using Python, R, or `AWK`: 

    
    awk 'BEGIN{OFS="\t"} {
        chrom=$1;
        start=$2-1;
        end=$3;
        name="junction";
        score=$6;
        strand=($4==1?"+":($4==2?"-":"."));
        blockCount=2;
        # simplified block sizes
        blockSizes="1,1";
        blockStarts="0,"end-start-1;
        print chrom,start,end,name,score,strand,start,end,0,blockCount,blockSizes,blockStarts;
    }' SJ.out.tab > SJ.bed


## Using the IGV Genome Browser: 

This section describes the essential skills needed to explore RNA-Seq data in IGV, including navigation, loading files, adjusting display modes, coloring alignments, and filtering reads.

**1. Navigation**

When IGV opens, it will look like this:
![image.png](images/M5P1_IGV_Usage_690d4a01.png)

Usually, the GRCh38/hg38 human genome is loaded, so we need to change this to TAIR 10 for our use. Navigate to `Genomes > Download a hosted Genome`:

![image.png](images/M5P1_IGV_Usage_bc168294.png)

Now, find the TAIR10 genome, and download the sequence and annotations. Load it in the top left, so we see the TAIR10 Chromosomes and reference track: 

![image.png](images/M5P1_IGV_Usage_9acbfa3e.png)



- Zoom In/Out
Use the mouse scroll wheel, trackpad pinch, or the zoom slider to move between genome-wide and base-level views. You can also click on a chromosome to zoom in.**Click on chromosome 1 in the genome track, then right click RefSeq genes and switch from `collapsed` to `expanded`. We can look at different regions in the TAIR10 genome this way:

![image.png](images/M5P1_IGV_Usage_49dcf40f.png)


- Jump to a Gene or Coordinate
Type a gene name (for example, IRT1) or a coordinate range (for example, Chr4:10,705,378-10,710,159) into the IGV search bar. Click `Go`:

![image.png](images/M5P1_IGV_Usage_f08b2875.png)


**2. Loading Data**
First, we will need to use `samtools` to generate an index for our `bam` files. Open up a terminal window, load the environment `module_5_IGV`, navigate to your results within your student folder under `Module_5/m5_project_STAR/results/alignments/` and then run the command `samtools index` on our samples, CRN1-1 and ORN1-1. 

    ml mamba

    source activate module_5_IGV

    cd MyStudentFolder/Module_5/m5_project_STAR/results/alignments/

    samtools index CRN1-1_Aligned.sortedByCoord.bam

    samtools index ORN1-1_Aligned.sortedByCoord.bam

Then, we will need to make our GTF file readable by IGV. To do this, we need to sort the annotation. Navigate to the annotation folder and run: 
    
    sort -k1,1 -k4,4n Arabidopsis_thaliana.TAIR10.62.gtf > Arabidopsis_thaliana.TAIR10.62.sorted.gtf

Next, we will take the sorted genome and compress it to `.bg.gz`

    bgzip Arabidopsis_thaliana.TAIR10.62.sorted.gtf

Finally, we index with `tabix`. Indexing the GTF file allows IGV to navigate it quickly and keeps the program running smoothly: 

    tabix -p gff Arabidopsis_thaliana.TAIR10.62.sorted.gtf.gz
    
Load Alignment or Coverage Files
Use File → Load from File to open:

`*.bam` (aligned reads)

`*.bam.bai` (index file)

`*.bigwig` (coverage tracks)

`*.bed` or `*.bedgraph`

`*.gtf` or `*.gff3` (annotation)

*Load Multiple Samples*
Open several BAM or bigWig files to compare conditions, such as CRN1-1 and ORN1-1. Click File > Load from File...
![image.png](images/M5P1_IGV_Usage_617321a4.png)


We will load the two samples as well as the GTF file used for annotation: 

*[Screenshot omitted from the public version (it showed account-specific details).]*


Hint: use the compressed and sorted GTF file in `m5_project_STAR/reference/annotation/`

*[Screenshot omitted from the public version (it showed account-specific details).]*


Now we should see our 2 samples' tracks, their splice junctions, and the `gtf` file at the bottom near the RefSeq TAIR10 genome. 

![image.png](images/M5P1_IGV_Usage_6b48a6af.png)

Here we are looking at the reads for IRT1 on both the CRN1-1 and ORN1-1 samples.

## You can right-click any track to change its display mode, color alignments, or filter reads: 

![image.png](images/M5P1_IGV_Usage_f51aa38b.png)

**3. Setting Track Display Mode**

- Collapsed
Shows a compact pileup of reads. Good for high-level coverage inspection.

- Expanded
Displays each read individually. Useful for inspecting mismatches, splice junctions, strandedness, or insert sizes.

- Squished
A compressed version of the expanded view. Useful when visualizing many samples.

To change modes:
Right-click a track → Collapsed / Expanded / Squished.

**4. Coloring Alignments**

- By Strand
Highlights transcription direction and checks strandedness of RNA-Seq libraries.

- By Read Group
Colors reads based on sequencing lane/library information.

- By Insert Size
Highlights abnormal fragment lengths and potential artifacts.

- To apply coloring:
Right-click a track → Color Alignments By.

**5. Filtering Reads**

- Hide Secondary or Supplementary Alignments
Removes multi-mapped or complex split reads from view.

- Hide Duplicates
Useful when PCR duplicates distort coverage.

- Adjust Mapping Quality (MAPQ) Threshold
Right-click a track → Filter and adjust the MAPQ slider to reveal low-confidence alignments.


## Viewing alignments: 

To look at our alignments closely, we can use the IRT1 gene as a reference: 

![image.png](images/M5P1_IGV_Usage_eaabea99.png)


When looking at the reads, it is important to note the read quality and mismatches: 

*[Annotated IGV tutorial slide omitted (third-party material).]*

*[Annotated IGV tutorial slide omitted (third-party material).]*

White reads have a mapping quality of 0, and low-quality base calls are semi-transparent:

*[Annotated IGV tutorial slide omitted (third-party material).]*

*[Annotated IGV tutorial slide omitted (third-party material).]*


**In IRT1 for CRN1-1, we can see several mismatches:**


![image.png](images/M5P1_IGV_Usage_b83b1e8b.png)


#### Looking at alignments via Zoom: 
*[Annotated IGV tutorial slide omitted (third-party material).]*

*[Annotated IGV tutorial slide omitted (third-party material).]*

Down-sampled reads are reads that have such high coverage, IGV caps the reads so that it saves memory: 

*[Annotated IGV tutorial slide omitted (third-party material).]*

In our experiment, IRT1 is down sampled in ORN1-1: 

![image.png](images/M5P1_IGV_Usage_abd195bb.png)


Why might that be?


#### Splice Junctions: 

We can group our splice junctions by strand: 
![image.png](images/M5P1_IGV_Usage_32b9862a.png)

Junctions on the + strand are in red, and junctions on the - strand are in blue. We can see the content of these splice junctions by their height: ie, a larger representation on the splice junction map means more reads are spliced at that location: 
*[Annotated IGV tutorial slide omitted (third-party material).]*
*[Annotated IGV tutorial slide omitted (third-party material).]*

This means if a gene has many reads spanning exons, the junction will be larger: 
*[Annotated IGV tutorial slide omitted (third-party material).]*

And if there are few reads, the junction will be much smaller: 

*[Annotated IGV tutorial slide omitted (third-party material).]*

Let's look at the splicing for IRT1

![image.png](images/M5P1_IGV_Usage_63f64193.png)

When a read spans a splice junction, it is connected with a blue line: 

*[Annotated IGV tutorial slide omitted (third-party material).]*

We can see that reads span exons frequently in IRT1 within ORN1-1: 

![image.png](images/M5P1_IGV_Usage_5ded0f79.png)

We can model these splice junctions with a **Shashimi plot**: 

*[Annotated IGV tutorial slide omitted (third-party material).]*

*[Annotated IGV tutorial slide omitted (third-party material).]*

*[Annotated IGV tutorial slide omitted (third-party material).]*

*[Annotated IGV tutorial slide omitted (third-party material).]*

You can read more about Shashimi plots here: [Quantitative visualization of alternative exon expression from RNA-seq data](https://academic.oup.com/bioinformatics/article/31/14/2400/254271)

To do this with our samples at IRT1, we right-click the splice junction tracks and click Shashimi Plot:

![image.png](images/M5P1_IGV_Usage_62b7c704.png)

This is the Shashimi Plot for IRT1 across CRN1-1 and ORN1-1
![image.png](images/M5P1_IGV_Usage_9757a969.png)

We will stop here for this practice, and we will continue in the next one. 

## Further IGV learning

<iframe width="560" height="315" src="https://www.youtube.com/embed/E_G8z_2gTYM?si=jpFvIZpgsVJ4F_MR" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

<iframe width="560" height="315" src="https://www.youtube.com/embed/eZUI6lcIX3I?si=vVP-Z8t589MUHU78" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

<iframe width="560" height="315" src="https://www.youtube.com/embed/awGN-rpLYas?si=zpTy-bnCS5QTfoi_" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

<iframe width="560" height="315" src="https://www.youtube.com/embed/dx9-N8b9Yj4?si=OJRRM3DPPCgz1BVy" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>